# Row-Subset Single-Tree Contract for GBNet

This notebook is a readable, user-facing version of the contract test.

It does three things:
1. Shows the reference behavior we want from native XGBoost and LightGBM when a single tree is trained on only a chosen subset of rows.
2. Confirms that the existing GBNet full-data one-tree path still matches native full-data training.
3. Checks whether `gbnet` can reproduce the stricter subset-loss contract through a future `gb_step(row_indices=...)` API.

Today, the expected result is that the native reference sections work, while the final GBNet status table reports that the API is still missing.

In [ ]:
import lightgbm as lgb
import numpy as np
import pandas as pd
import torch
import xgboost as xgb

from gbnet import lgbmodule as lgm
from gbnet import xgbmodule as xgm

np.set_printoptions(suppress=True, precision=6)
torch.set_printoptions(sci_mode=False, precision=6)

## Small fixture dataset

The selected training rows are `0, 2, 4, 6`.

Those rows follow a mild increasing pattern. The held-out rows have large positive or negative targets. That makes the dataset useful for this contract, because a tree trained on all rows should behave differently from a tree trained on only the selected rows.

In [ ]:
X = np.arange(8, dtype=np.float32).reshape(-1, 1)
y = np.array(
    [0.0, 100.0, 1.0, 100.0, 2.0, -100.0, 3.0, -100.0],
    dtype=np.float32,
).reshape(-1, 1)

row_indices = np.array([0, 2, 4, 6], dtype=np.int64)
heldout_rows = np.setdiff1d(np.arange(X.shape[0]), row_indices)

fixture = pd.DataFrame(
    {
        "row": np.arange(X.shape[0]),
        "feature": X.flatten(),
        "target": y.flatten(),
        "used_for_training": np.isin(np.arange(X.shape[0]), row_indices),
    }
)
fixture

## Native backend reference helpers

Each helper below trains exactly one tree on only the selected rows, then predicts for all rows.

That is the behavior the future GBNet feature needs to reproduce.

In [ ]:
xgb_params = {
    "eta": 1.0,
    "max_depth": 2,
    "lambda": 0.0,
    "alpha": 0.0,
    "gamma": 0.0,
    "min_child_weight": 0.0,
    "tree_method": "hist",
    "nthread": 1,
}

lgb_params = {
    "learning_rate": 1.0,
    "num_leaves": 4,
    "max_depth": 2,
    "min_data_in_leaf": 1,
    "min_sum_hessian_in_leaf": 0.0,
    "lambda_l1": 0.0,
    "lambda_l2": 0.0,
    "verbose": -1,
    "verbosity": -1,
    "num_threads": 1,
    "seed": 0,
}


def xgb_reference_predictions(X, y, row_indices, params):
    booster = xgb.train(
        {
            **params,
            "objective": "reg:squarederror",
            "base_score": 0.0,
        },
        xgb.DMatrix(X[row_indices], label=y[row_indices].reshape(-1)),
        num_boost_round=1,
    )
    return booster.predict(xgb.DMatrix(X)).reshape(-1, 1)


def lgb_reference_predictions(X, y, row_indices, params):
    booster = lgb.train(
        {
            **params,
            "objective": "regression",
            "boost_from_average": False,
        },
        lgb.Dataset(
            X[row_indices],
            label=y[row_indices].reshape(-1),
            init_score=np.zeros(len(row_indices), dtype=np.float32),
            free_raw_data=False,
        ),
        num_boost_round=1,
    )
    return booster.predict(X).reshape(-1, 1)

In [ ]:
comparison = fixture.copy()

comparison["xgb_subset_pred"] = xgb_reference_predictions(X, y, row_indices, xgb_params).flatten()
comparison["xgb_full_pred"] = xgb_reference_predictions(X, y, np.arange(X.shape[0]), xgb_params).flatten()
comparison["xgb_abs_diff"] = np.abs(comparison["xgb_subset_pred"] - comparison["xgb_full_pred"])

comparison["lgb_subset_pred"] = lgb_reference_predictions(X, y, row_indices, lgb_params).flatten()
comparison["lgb_full_pred"] = lgb_reference_predictions(X, y, np.arange(X.shape[0]), lgb_params).flatten()
comparison["lgb_abs_diff"] = np.abs(comparison["lgb_subset_pred"] - comparison["lgb_full_pred"])

comparison

## Sanity checks for the fixture

These numbers should be meaningfully above zero.

If they are, the fixture is informative: training on the subset is measurably different from training on all rows, and the subset-trained tree still produces predictions for held-out rows.

In [ ]:
reference_summary = pd.DataFrame(
    [
        {
            "backend": "xgboost",
            "max_abs_diff_subset_vs_full": float(comparison["xgb_abs_diff"].max()),
            "max_abs_pred_on_heldout_rows": float(np.abs(comparison.loc[heldout_rows, "xgb_subset_pred"]).max()),
        },
        {
            "backend": "lightgbm",
            "max_abs_diff_subset_vs_full": float(comparison["lgb_abs_diff"].max()),
            "max_abs_pred_on_heldout_rows": float(np.abs(comparison.loc[heldout_rows, "lgb_subset_pred"]).max()),
        },
    ]
)

reference_summary

## GBNet contract check

This cell uses the same setup as the test suite.

It checks two scenarios for each backend:
- `full_training_baseline`: train on all rows and verify that the existing GBNet path still matches native full-data one-tree training.
- `subset_loss_plus_row_indices`: train-time forward still uses the full `X`, but the loss only uses `preds[row_indices]` and `y[row_indices]`, then GBNet is asked to update with `gb_step(row_indices=row_indices)`.

Interpretation:
- `pass`: GBNet matched the native backend for that scenario.
- `missing_support`: this checkout of `gbnet` does not yet support the subset-loss contract.
- `mismatch`: GBNet ran, but its predictions did not match the native backend.

In [ ]:
def check_gbnet_full_contract(module_class, backend_name, params, expected_full_preds, loss_scale):
    module = module_class(
        batch_size=X.shape[0],
        input_dim=X.shape[1],
        output_dim=1,
        params=params,
    )
    loss_fn = torch.nn.MSELoss()

    module.train()
    preds = module(X)
    loss = loss_scale * loss_fn(preds, torch.tensor(y, dtype=torch.float))
    loss.backward(create_graph=True)
    module.gb_step()

    module.eval()
    actual_preds = module(X).detach().numpy()
    max_abs_error = float(np.max(np.abs(actual_preds - expected_full_preds)))
    status = "pass" if np.allclose(actual_preds, expected_full_preds, rtol=1e-6, atol=1e-6) else "mismatch"

    return {
        "backend": backend_name,
        "scenario": "full_training_baseline",
        "status": status,
        "max_abs_error": max_abs_error,
        "details": "" if status == "pass" else "Predictions did not match native full-data training.",
    }


def check_gbnet_subset_contract(module_class, backend_name, params, expected_subset_preds, loss_scale):
    module = module_class(
        batch_size=X.shape[0],
        input_dim=X.shape[1],
        output_dim=1,
        params=params,
    )
    loss_fn = torch.nn.MSELoss()

    try:
        module.train()
        preds = module(X)
        loss = loss_scale * loss_fn(preds[row_indices], torch.tensor(y[row_indices], dtype=torch.float))
        loss.backward(create_graph=True)
        module.gb_step(row_indices=row_indices)
    except Exception as exc:
        return {
            "backend": backend_name,
            "scenario": "subset_loss_plus_row_indices",
            "status": "missing_support",
            "max_abs_error": np.nan,
            "details": f"{type(exc).__name__}: {exc}",
        }

    module.eval()
    actual_preds = module(X).detach().numpy()
    max_abs_error = float(np.max(np.abs(actual_preds - expected_subset_preds)))
    status = "pass" if np.allclose(actual_preds, expected_subset_preds, rtol=1e-6, atol=1e-6) else "mismatch"

    return {
        "backend": backend_name,
        "scenario": "subset_loss_plus_row_indices",
        "status": status,
        "max_abs_error": max_abs_error,
        "details": "" if status == "pass" else "Predictions did not match native subset training.",
    }


gbnet_status = pd.DataFrame(
    [
        check_gbnet_full_contract(
            xgm.XGBModule,
            "xgboost",
            xgb_params,
            comparison[["xgb_full_pred"]].to_numpy(),
            loss_scale=0.5,
        ),
        check_gbnet_subset_contract(
            xgm.XGBModule,
            "xgboost",
            xgb_params,
            comparison[["xgb_subset_pred"]].to_numpy(),
            loss_scale=0.5,
        ),
        check_gbnet_full_contract(
            lgm.LGBModule,
            "lightgbm",
            lgb_params,
            comparison[["lgb_full_pred"]].to_numpy(),
            loss_scale=1.0,
        ),
        check_gbnet_subset_contract(
            lgm.LGBModule,
            "lightgbm",
            lgb_params,
            comparison[["lgb_subset_pred"]].to_numpy(),
            loss_scale=1.0,
        ),
    ]
)

gbnet_status

In [ ]:
full_statuses = set(gbnet_status.loc[gbnet_status["scenario"] == "full_training_baseline", "status"])
subset_statuses = set(gbnet_status.loc[gbnet_status["scenario"] == "subset_loss_plus_row_indices", "status"])

if full_statuses != {"pass"}:
    raise AssertionError("The existing full-data baseline no longer matches native one-tree training.")
elif subset_statuses == {"pass"}:
    print("GBNet row-subset single-tree training is working for both backends.")
elif "missing_support" in subset_statuses:
    print("The existing full-data baseline is working.")
    print("This checkout does not yet support the stricter subset-loss plus row_indices contract.")
    print("After implementing the feature, rerun the notebook and the subset rows should switch to 'pass'.")
else:
    raise AssertionError("GBNet ran the subset contract, but it did not match native subset-trained predictions.")